# Build `meas_node_binaries.tgz`

This notebook provisions a single throwaway FABRIC VM, sets up an `mfuser` account on it, runs
[`build_meas_node_binaries.sh`](build_meas_node_binaries.sh) to (re)build the ansible bootstrap bundle, and downloads
the resulting `meas_node_binaries.tgz` back to this notebook's directory.

Rebuild this whenever the real meas-node base image's default Ubuntu/Python version changes. The tarball bakes in a
`python3.X` site-packages path at build time, so a version drift between the VM built here and the real meas-node
image is exactly what breaks `ansible-playbook` with a `ModuleNotFoundError` at bootstrap time.

Pick an **Ubuntu** image below -- `build_meas_node_binaries.sh` shells out to `apt`, so a non-Debian-based image
(Rocky, CentOS, ...) will fail.

In [ ]:
from pathlib import Path

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.show_config()

## Configuration

Edit `IMAGE_NAME` to pick which image the tarball gets built against. Run the "list available images" cell further
down first if you need to check the exact image name string -- these change over time.

In [ ]:
# --- Edit these -------------------------------------------------------------

# The FABRIC image to build the tarball on. Must be Ubuntu, and should match
# the Ubuntu/Python version used by the real meas-node image. Run
# list_available_images() below to see valid values.
IMAGE_NAME = "default_ubuntu_24"

# Leave as None to let fablib pick a site automatically, or set an explicit
# site name (see fablib.get_site_names()).
SITE = None

SLICE_NAME = "build-meas-node-binaries"
NODE_NAME = "meas-node-binaries-builder"

CORES = 4
RAM = 8
DISK = 20

# Repo this VM will clone as mf_git in order to run the build script from the
# same paths install_ansible.sh / bootstrap_docker.py expect on a real node.
MF_REPO_URL = "https://github.com/fabric-testbed/MeasurementFramework.git"
MF_REPO_BRANCH = "main"
BUILD_SCRIPT_REL_PATH = "instrumentize/experiment_bootstrap/build_meas_node_binaries.sh"
REMOTE_TARBALL_REL_PATH = "instrumentize/experiment_bootstrap/meas_node_binaries.tgz"

# Where to save the downloaded tarball -- defaults to this notebook's directory.
OUTPUT_DIR = Path.cwd()
LOCAL_TARBALL_PATH = OUTPUT_DIR / "meas_node_binaries.tgz"

print(f"Will build on image '{IMAGE_NAME}', download to {LOCAL_TARBALL_PATH}")

## Available images

Prints every image name fablib currently knows about, so you can double check the `IMAGE_NAME` value above.

In [ ]:
def list_available_images():
    """Prints all image names fablib knows about, with whatever metadata it has for each."""
    images = fablib.get_image_names()
    for name in sorted(images):
        print(f"{name}: {images[name]}")
    return images


available_images = list_available_images()

if IMAGE_NAME not in available_images:
    raise ValueError(
        f"IMAGE_NAME '{IMAGE_NAME}' is not one of the available images: "
        f"{sorted(available_images)}"
    )
else:
    print(f'\nThe chosen "{IMAGE_NAME}" is an available image.')

## Create the slice and node

In [ ]:
site = SITE or fablib.get_random_site()
print(f"Using site: {site}")

slice = fablib.new_slice(name=SLICE_NAME)

node = slice.add_node(
    name=NODE_NAME,
    site=site,
    image=IMAGE_NAME,
    cores=CORES,
    ram=RAM,
    disk=DISK,
)

slice.submit()

## Wait for SSH

In [ ]:
slice = fablib.get_slice(name=SLICE_NAME)
slice.wait_ssh(progress=True)
slice.post_boot_config()

node = slice.get_node(name=NODE_NAME)
print(f"Node management IP: {node.get_management_ip()}")

## Set up the `mfuser` account

Creates the `mfuser` account, grants it passwordless sudo (`build_meas_node_binaries.sh` -- and the real ansible
bootstrap it prepares -- both shell out to `sudo` non-interactively, so this can't be a password prompt), and clones
this repo to `/home/mfuser/mf_git` so the script can find `requirements.txt` / `requirements.yml` at the paths it
expects.

In [ ]:
setup_cmd = f"""
set -euo pipefail

if ! id mfuser >/dev/null 2>&1; then
    sudo useradd -m -s /bin/bash mfuser
fi

echo "mfuser ALL=(ALL) NOPASSWD:ALL" | sudo tee /etc/sudoers.d/mfuser >/dev/null
sudo chmod 440 /etc/sudoers.d/mfuser

sudo apt-get update
sudo apt-get install -y git

sudo -u mfuser mkdir -p /home/mfuser/mf_git
if [ ! -d /home/mfuser/mf_git/.git ]; then
    sudo -u mfuser git clone --branch {MF_REPO_BRANCH} {MF_REPO_URL} /home/mfuser/mf_git
else
    sudo -u mfuser git -C /home/mfuser/mf_git fetch origin {MF_REPO_BRANCH}
    sudo -u mfuser git -C /home/mfuser/mf_git checkout {MF_REPO_BRANCH}
    sudo -u mfuser git -C /home/mfuser/mf_git pull origin {MF_REPO_BRANCH}
fi

sudo chmod +x /home/mfuser/mf_git/{BUILD_SCRIPT_REL_PATH}
"""

stdout, stderr = node.execute(setup_cmd)
print(stdout)
print(stderr)

## Run `build_meas_node_binaries.sh` as `mfuser`

This does the live apt/pip/galaxy install and repackages `~/.local` + `~/.ansible` into the tarball -- expect this to
take a few minutes.

In [ ]:
build_cmd = f"sudo -u mfuser -H bash -lc 'bash /home/mfuser/mf_git/{BUILD_SCRIPT_REL_PATH}'"

stdout, stderr = node.execute(build_cmd)
print(stdout)
print(stderr)

## Stage the tarball for download

`node.download_file()` connects as the node's default management user (`node.get_username()`, e.g. `ubuntu`), not
`mfuser` -- and `/home/mfuser` isn't readable by that user. Copy the tarball out to the default user's home directory
first so it can actually be fetched.

In [ ]:
default_user = node.get_username()
staged_tarball_path = f"/home/{default_user}/meas_node_binaries.tgz"

stage_cmd = f"""
set -euo pipefail
sudo cp /home/mfuser/mf_git/{REMOTE_TARBALL_REL_PATH} {staged_tarball_path}
sudo chown {default_user}:{default_user} {staged_tarball_path}
sudo chmod 644 {staged_tarball_path}
"""

stdout, stderr = node.execute(stage_cmd)
print(stdout)
print(stderr)

## Download the tarball

Saves `meas_node_binaries.tgz` next to this notebook.

In [ ]:
node.download_file(str(LOCAL_TARBALL_PATH), staged_tarball_path)

assert LOCAL_TARBALL_PATH.exists(), "Download appears to have failed -- file not found locally."
print(f"Downloaded to {LOCAL_TARBALL_PATH} ({LOCAL_TARBALL_PATH.stat().st_size} bytes)")

## Clean up

Once you've confirmed the tarball downloaded correctly, delete the throwaway VM. Left commented out on purpose so
running the whole notebook top-to-bottom doesn't tear down the slice before you've had a chance to look at it.

In [ ]:
# fablib.delete_slice(SLICE_NAME)